Scenario 4 Dataset (Kaggle)

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch

# 1. Kaggle paths
experiment_dir = Path("/kaggle/working")
results_dir = experiment_dir / "scenario4_train_test_e4000"
results_dir.mkdir(parents=True, exist_ok=True)

# 2. Find uploaded files
data_files = list(
    Path("/kaggle/input").rglob("scenario4_wind_train_test.pt")
)

model_files = list(
    Path("/kaggle/input").rglob("02_climate_model.ipynb")
)

if len(data_files) != 1:
    raise FileNotFoundError(
        f"Expected one Scenario 4 dataset, found: {data_files}"
    )

if len(model_files) != 1:
    raise FileNotFoundError(
        f"Expected one model notebook, found: {model_files}"
    )

data_path = data_files[0]
model_notebook = model_files[0]

# 3. Load preprocessed dataset
dataset = torch.load(
    data_path,
    map_location="cpu",
    weights_only=True
)

train_scaled = dataset["train_scaled"].numpy()
test_scaled = dataset["test_scaled"].numpy()

location_indices = dataset["location_indices"]

# 4. Experiment configuration
seq_len = dataset["seq_len"]
patch_lengths = dataset["patch_lengths"]
horizons = dataset["horizons"]

d_model = dataset["d_model"]
epochs = dataset["epochs"]
learning_rate = dataset["learning_rate"]
lyap_weight = dataset["lyap_weight"]
seed = dataset["seed"]

input_dim = train_scaled.shape[1]

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# 5. Build sliding windows
def build_windows(data, seq_len, horizon):
    X, Y = [], []

    for i in range(len(data) - seq_len - horizon):
        X.append(data[i:i + seq_len])
        Y.append(
            data[i + seq_len:i + seq_len + horizon]
            .T.reshape(-1)
        )

    x = torch.tensor(np.stack(X), dtype=torch.float32)
    y = torch.tensor(np.stack(Y), dtype=torch.float32)

    return x, y

# 6. Test with H=10
x_train, y_train = build_windows(train_scaled, seq_len, 10)
x_test, y_test = build_windows(test_scaled, seq_len, 10)

# 7. Validate
assert device.type == "cuda", "Enable Kaggle GPU"
assert train_scaled.shape == (2336, 5)
assert test_scaled.shape == (584, 5)
assert seq_len == 120
assert patch_lengths == [40, 50, 60, 70, 80, 90, 100]
assert horizons == [2, 4, 6, 8, 10, 12, 14, 16]
assert d_model == 48
assert epochs == 4000

assert np.isfinite(train_scaled).all()
assert np.isfinite(test_scaled).all()

assert x_train.shape == (2206, 120, 5)
assert y_train.shape == (2206, 50)
assert x_test.shape == (454, 120, 5)
assert y_test.shape == (454, 50)

# 8. Summary
print("=== SCENARIO 4 - KAGGLE ===")
print("GPU:", torch.cuda.get_device_name(0))
print("Dataset:", data_path)
print("Model notebook:", model_notebook)

print("\nData:")
print("Train:", train_scaled.shape)
print("Test:", test_scaled.shape)
print("Locations:", location_indices)

print("\nSliding windows (H=10):")
print("X_train:", x_train.shape)
print("Y_train:", y_train.shape)
print("X_test:", x_test.shape)
print("Y_test:", y_test.shape)

print("\nExperiment:")
print("Sequence length:", seq_len)
print("Patch lengths:", patch_lengths)
print("Horizons:", horizons)
print("d_model:", d_model)
print("Epochs:", epochs)
print("Total training runs:",
      len(patch_lengths) * len(horizons) * 4)

print("\nResults:", results_dir)


=== SCENARIO 4 - KAGGLE ===
GPU: Tesla T4
Dataset: /kaggle/input/datasets/tunnghanguynng/scenario4/scenario4_wind_train_test.pt
Model notebook: /kaggle/input/datasets/tunnghanguynng/climate/02_climate_model.ipynb

Data:
Train: (2336, 5)
Test: (584, 5)
Locations: [0, 5000, 10000, 15000, 20000]

Sliding windows (H=10):
X_train: torch.Size([2206, 120, 5])
Y_train: torch.Size([2206, 50])
X_test: torch.Size([454, 120, 5])
Y_test: torch.Size([454, 50])

Experiment:
Sequence length: 120
Patch lengths: [40, 50, 60, 70, 80, 90, 100]
Horizons: [2, 4, 6, 8, 10, 12, 14, 16]
d_model: 48
Epochs: 4000
Total training runs: 224

Results: /kaggle/working/scenario4_train_test_e4000


Load DeepKoopFormer Models (Scenario 4)

In [4]:
import ast
import json
import torch

# 1. Extract model definitions without running local dataset code
with open(model_notebook, "r", encoding="utf-8") as f:
    notebook = json.load(f)

allowed_nodes = (
    ast.Import,
    ast.ImportFrom,
    ast.FunctionDef,
    ast.ClassDef
)

definitions = []

for cell in notebook["cells"]:
    if cell["cell_type"] != "code":
        continue

    source = cell["source"]
    if isinstance(source, list):
        source = "".join(source)

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in tree.body:
        if isinstance(node, allowed_nodes):
            definitions.append(node)

module = ast.Module(body=definitions, type_ignores=[])
ast.fix_missing_locations(module)

exec(
    compile(module, str(model_notebook), "exec"),
    globals()
)

# 2. Model factory: seq_len fixed, patch_len variable
model_names = ["LSTM", "PatchTST", "Autoformer", "Informer"]

def create_model(model_name, patch_len, horizon):
    if model_name == "LSTM":
        return SimpleLSTMForecaster(
            input_dim=5,
            hidden_dim=48,
            num_layers=2,
            horizon=horizon
        )

    if model_name == "PatchTST":
        return Koopformer_PatchTST(
            input_dim, seq_len, horizon,
            patch_len=patch_len,
            d_model=d_model
        )

    if model_name == "Autoformer":
        return Koopformer_Autoformer(
            input_dim, seq_len, horizon,
            patch_len=patch_len,
            d_model=d_model
        )

    if model_name == "Informer":
        return Koopformer_Informer(
            input_dim, seq_len, horizon,
            patch_len=patch_len,
            d_model=d_model
        )

    raise ValueError(f"Unknown model: {model_name}")

# 3. Test forward pass at representative configurations
print("=== SCENARIO 4 - MODEL VALIDATION ===")

for p, h in [(40, 8), (70, 10), (100, 16)]:
    print(f"\nPatch length={p}, Horizon={h}")

    x_sample = torch.randn(2, seq_len, input_dim).to(device)

    for name in model_names:
        model = create_model(name, p, h).to(device)
        model.eval()

        with torch.no_grad():
            prediction = model(x_sample)

        expected_shape = (2, input_dim * h)
        assert prediction.shape == expected_shape, (
            f"{name}: {prediction.shape} != {expected_shape}"
        )

        n_params = sum(
            param.numel() for param in model.parameters()
        )

        print(
            f"{name:<12} "
            f"Output={tuple(prediction.shape)} "
            f"Params={n_params:,}"
        )

        del model

print("\nAll model configurations validated.")


=== SCENARIO 4 - MODEL VALIDATION ===

Patch length=40, Horizon=8
LSTM         Output=(2, 40) Params=31,336
PatchTST     Output=(2, 40) Params=73,192
Autoformer   Output=(2, 40) Params=53,552
Informer     Output=(2, 40) Params=73,144

Patch length=70, Horizon=10
LSTM         Output=(2, 50) Params=31,826
PatchTST     Output=(2, 50) Params=80,882
Autoformer   Output=(2, 50) Params=55,116
Informer     Output=(2, 50) Params=80,834

Patch length=100, Horizon=16
LSTM         Output=(2, 80) Params=33,296
PatchTST     Output=(2, 80) Params=89,552
Autoformer   Output=(2, 80) Params=60,144
Informer     Output=(2, 80) Params=89,504

All model configurations validated.


Training and Evaluation Functions

In [5]:
import torch.nn as nn
import torch.optim as optim

# 1. Reproducibility
def set_seed(seed=7):
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(seed)

# 2. Access nested model attributes
def _nested_getattr(obj, path):
    for p in path.split("."):
        obj = getattr(obj, p, None)
        if obj is None:
            return None
    return obj

# 3. Train on training windows only
def train_scenario4(
    model, x_train, y_train,
    epochs=4000, lr=3e-4,
    koop_attr="koop", lyap_weight=0.1
):
    model.to(device)
    x_train = x_train.to(device)
    y_train = y_train.to(device)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    mse = nn.MSELoss()

    losses, eigs = [], []

    # Lyapunov matrix P = I
    with torch.no_grad():
        try:
            _, z, z_next = model(
                x_train[:1], return_latents=True
            )
        except ValueError:
            z = z_next = torch.zeros(
                1, model.fc.in_features, device=device
            )

    P = torch.eye(z.shape[1], device=device)

    for ep in range(epochs):
        model.train()
        optimizer.zero_grad()

        pred, z, z_next = model(
            x_train, return_latents=True
        )

        pred_loss = mse(pred, y_train)

        if lyap_weight:
            zp = torch.einsum("bi,ij->bj", z, P)
            znp = torch.einsum("bi,ij->bj", z_next, P)

            lyap = torch.relu(
                (znp * z_next).sum(1)
                - (zp * z).sum(1)
            ).mean()
        else:
            lyap = torch.zeros(1, device=device)

        loss = pred_loss + lyap_weight * lyap

        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        # Track Koopman singular-value parameter
        obj = (
            _nested_getattr(model, koop_attr)
            if koop_attr else None
        )

        eig_raw = (
            obj.S_raw
            if isinstance(obj, StrictStableKoopmanOperator)
            else None
        )

        eigs.append(
            0.0 if eig_raw is None
            else torch.sigmoid(eig_raw).abs().max().item()
        )

        if ep % max(1, epochs // 10) == 0 or ep == epochs - 1:
            print(
                f"Epoch {ep:>4}/{epochs} "
                f"MSE {pred_loss.item():.5e} "
                f"Lyap {(lyap_weight * lyap).item():.5e}"
            )

    # Predictions on training windows
    model.eval()

    with torch.no_grad():
        pred_train = model(x_train).cpu().numpy()

    return (
        pred_train,
        np.asarray(losses),
        np.asarray(eigs)
    )

# 4. Evaluate on unseen test windows
def evaluate_scenario4(model, x_test, y_test):
    model.to(device)
    model.eval()

    with torch.no_grad():
        pred_test = model(x_test.to(device)).cpu().numpy()

    truth_test = y_test.cpu().numpy()

    error = pred_test - truth_test

    mse_value = float(np.mean(error ** 2))
    mae_value = float(np.mean(np.abs(error)))

    return pred_test, mse_value, mae_value

print("=== SCENARIO 4 FUNCTIONS READY ===")
print("Device:", device)
print("Epochs:", epochs)
print("Learning rate:", learning_rate)
print("Lyapunov weight:", lyap_weight)
print("Models:", model_names)


=== SCENARIO 4 FUNCTIONS READY ===
Device: cuda
Epochs: 4000
Learning rate: 0.0003
Lyapunov weight: 0.1
Models: ['LSTM', 'PatchTST', 'Autoformer', 'Informer']


Scenario 4 Grid Search & Checkpointing

In [6]:
import gc
import hashlib
import itertools
import shutil
import pandas as pd
import numpy as np
import torch

# 1. Configuration
model_names = ["LSTM", "PatchTST", "Autoformer", "Informer"]
metrics_path = results_dir / "metrics.csv"

# Number of models to train per execution
RUNS_PER_BATCH = 224

configs = list(itertools.product(
    patch_lengths, horizons, model_names
))

assert device.type == "cuda"
assert epochs == 4000
assert seq_len == 120
assert d_model == 48
assert len(configs) == 224

results_dir.mkdir(parents=True, exist_ok=True)

# 2. Dataset fingerprint to prevent mixing different datasets
data_hash = hashlib.sha256(
    train_scaled.tobytes() + test_scaled.tobytes()
).hexdigest()

def checkpoint_path(p, h, name):
    return results_dir / f"{name.lower()}_p{p}_h{h}_d{d_model}.pt"

def record(name, p, h, set_name, mse, mae):
    return {
        "Model": name,
        "PatchLen": p,
        "Horizon": h,
        "d_model": d_model,
        "Set": set_name,
        "Epochs": epochs,
        "MSE": float(mse),
        "MAE": float(mae)
    }

def save_metrics(records):
    df = pd.DataFrame(
        records,
        columns=[
            "Model", "PatchLen", "Horizon", "d_model",
            "Set", "Epochs", "MSE", "MAE"
        ]
    )
    temp = metrics_path.with_suffix(".csv.tmp")
    df.to_csv(temp, index=False)
    temp.replace(metrics_path)

# 3. Recover checkpoints saved as a previous Kaggle Dataset
resume_dirs = [
    p for p in Path("/kaggle/input").rglob(
        "scenario4_train_test_e4000"
    )
    if p.is_dir()
]

if len(resume_dirs) > 1:
    raise RuntimeError("Multiple checkpoint sources found.")

if resume_dirs:
    for source in resume_dirs[0].glob("*_p*_h*_d*.pt"):
        destination = results_dir / source.name
        if not destination.exists():
            shutil.copy2(source, destination)

# 4. Restore completed runs and RNG
set_seed(seed)

records = []
completed = 0

for p, h, name in configs:
    path = checkpoint_path(p, h, name)

    if not path.is_file():
        break

    ckpt = torch.load(
        path, map_location="cpu", weights_only=True
    )

    assert ckpt["model"] == name
    assert ckpt["patch_len"] == p
    assert ckpt["horizon"] == h
    assert ckpt["seq_len"] == seq_len
    assert ckpt["d_model"] == d_model
    assert ckpt["epochs"] == epochs
    assert ckpt["data_hash"] == data_hash

    records.append(record(
        name, p, h, "Train",
        ckpt["train_mse"], ckpt["train_mae"]
    ))
    records.append(record(
        name, p, h, "Test",
        ckpt["test_mse"], ckpt["test_mae"]
    ))

    completed += 1

    torch.set_rng_state(ckpt["rng_torch"])
    torch.cuda.set_rng_state_all(ckpt["rng_cuda"])

# Stop if checkpoints are non-sequential
for p, h, name in configs[completed:]:
    if checkpoint_path(p, h, name).exists():
        raise RuntimeError(
            "Non-sequential checkpoints detected."
        )

save_metrics(records)

# 5. Select the next batch
batch_configs = configs[
    completed:completed + RUNS_PER_BATCH
]

print("=== SCENARIO 4 GRID SEARCH ===")
print("GPU:", torch.cuda.get_device_name(0))
print("Total runs:", len(configs))
print("Completed:", completed)
print("This batch:", len(batch_configs))
print("Remaining:", len(configs) - completed)
print("Epochs:", epochs)
print("Results:", results_dir)

# 6. Train the selected configurations
for run_idx, (p, h, name) in enumerate(
    batch_configs, start=completed + 1
):
    print(f"\n{'=' * 60}")
    print(
        f"[{run_idx}/224] {name} | "
        f"PatchLen={p}, Horizon={h}"
    )
    print(f"{'=' * 60}")

    # Build train/test windows for this horizon
    x_train, y_train = build_windows(
        train_scaled, seq_len, h
    )
    x_test, y_test = build_windows(
        test_scaled, seq_len, h
    )

    # Initialize model
    model = create_model(name, p, h)

    is_lstm = name == "LSTM"

    # Train on train set only
    pred_train, loss, eig = train_scenario4(
        model,
        x_train,
        y_train,
        epochs=epochs,
        lr=learning_rate,
        koop_attr=None if is_lstm else "koop",
        lyap_weight=0.0 if is_lstm else lyap_weight
    )

    # Train metrics
    truth_train = y_train.numpy()
    train_error = pred_train - truth_train

    train_mse = float(np.mean(train_error ** 2))
    train_mae = float(np.mean(np.abs(train_error)))

    # Test evaluation (no weight updates)
    pred_test, test_mse, test_mae = evaluate_scenario4(
        model, x_test, y_test
    )

    truth_test = y_test.numpy()

    print(
        f"Train: MSE={train_mse:.6f}, "
        f"MAE={train_mae:.6f}"
    )
    print(
        f"Test:  MSE={test_mse:.6f}, "
        f"MAE={test_mae:.6f}"
    )

    # 7. Save complete checkpoint
    checkpoint = {
        "model": name,
        "model_state_dict": model.cpu().state_dict(),

        "pred_train": torch.from_numpy(pred_train.copy()),
        "truth_train": torch.from_numpy(truth_train.copy()),

        "pred_test": torch.from_numpy(pred_test.copy()),
        "truth_test": torch.from_numpy(truth_test.copy()),

        "loss": torch.from_numpy(loss.copy()),
        "eig": torch.from_numpy(eig.copy()),

        "train_mse": train_mse,
        "train_mae": train_mae,
        "test_mse": test_mse,
        "test_mae": test_mae,

        "patch_len": p,
        "horizon": h,
        "seq_len": seq_len,
        "d_model": d_model,
        "input_dim": input_dim,
        "epochs": epochs,
        "data_hash": data_hash,

        "rng_torch": torch.get_rng_state(),
        "rng_cuda": torch.cuda.get_rng_state_all()
    }

    path = checkpoint_path(p, h, name)
    temp_path = path.with_suffix(".pt.tmp")

    torch.save(checkpoint, temp_path)
    temp_path.replace(path)

    # 8. Update metrics
    records.append(record(
        name, p, h, "Train", train_mse, train_mae
    ))
    records.append(record(
        name, p, h, "Test", test_mse, test_mae
    ))

    save_metrics(records)

    print("Checkpoint saved:", path)
    print("Completed:", run_idx, "/ 224")

    # Release GPU memory
    del (
        model, x_train, y_train, x_test, y_test,
        pred_train, pred_test, truth_train,
        truth_test, train_error, loss, eig, checkpoint
    )

    gc.collect()
    torch.cuda.empty_cache()

# 9. Package results for download / recovery
archive = shutil.make_archive(
    str(results_dir),
    "zip",
    root_dir=results_dir.parent,
    base_dir=results_dir.name
)

print("\n=== BATCH COMPLETE ===")
print("Completed:", completed + len(batch_configs), "/ 224")
print("Checkpoint folder:", results_dir)
print("Backup archive:", archive)


=== SCENARIO 4 GRID SEARCH ===
GPU: Tesla T4
Total runs: 224
Completed: 0
This batch: 224
Remaining: 224
Epochs: 4000
Results: /kaggle/working/scenario4_train_test_e4000

[1/224] LSTM | PatchLen=40, Horizon=2
Epoch    0/4000 MSE 1.50701e-01 Lyap 0.00000e+00
Epoch  400/4000 MSE 1.76731e-02 Lyap 0.00000e+00
Epoch  800/4000 MSE 1.12169e-02 Lyap 0.00000e+00


KeyboardInterrupt: 